<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Kaggel_Script/master-ply-renderer-kaggle-tpu-cpu-mode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://kaggle.com/kernels/welcome?src=https://github.com/Mahendra2409/PyBlender/blob/main/Kaggel_Script/master-ply-renderer-kaggle-tpu-cpu-mode.ipynb" target="_parent"><img src="https://img.shields.io/badge/Open_in-Kaggle-20BEFF?logo=kaggle&logoColor=white" alt="Open In Kaggle"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

# 🎨 Master PLY Renderer (Kaggle TPU — CPU Mode)

Renders `.ply` format 3D meshes with **ceramic material** using Blender Cycles on Kaggle's CPU on TPU runtime (~96GB RAM).

### Adding New Datasets
1. Duplicate any config cell (4.x)
2. Update transforms, camera, lighting for your new mesh
3. Uncomment the new key in `RENDER_TYPES` (Cell 5)


In [ ]:
# @title 0. WandB Authentication
!pip install wandb -q
import wandb
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api)

In [ ]:
# @title 1. Install Dependencies & Toolbox
!wget -nc -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz

!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip --upgrade > /dev/null 2>&1
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install \
    scipy matplotlib numpy wandb blendertoolbox plotly google-cloud-storage plyfile \
    -q --no-input --disable-pip-version-check

!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
!mv BlenderToolbox/blendertoolbox /kaggle/working/

In [ ]:
# Install dependency
!pip install -q google-cloud-storage

# Authenticate Google Cloud Storage
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
gcs_key_json = user_secrets.get_secret("GCS_SERVICE_ACCOUNT_KEY")

GCS_KEY_PATH = "/tmp/gcs_service_account.json"

with open(GCS_KEY_PATH, "w") as f:
    f.write(gcs_key_json)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCS_KEY_PATH

print(f"GCS credentials saved to {GCS_KEY_PATH}")

# Import GCS client
from google.cloud import storage

# Connect
client = storage.Client()

bucket = client.bucket("pyblender-render-farm")

try:
    blobs = list(bucket.list_blobs(max_results=1))
    print("SUCCESS: Connected to gs://pyblender-render-farm")
except Exception as e:
    print(f"WARNING: Could not verify bucket: {e}")

In [ ]:
#@title 3. Initialize Config Registries
PLY_CONFIGS = {}
print("Config registries initialized.")

In [ ]:
#@title 4.1 Config — armadillo_PLY
PLY_CONFIGS["armadillo_PLY"] = {
    "SAVE_BLEND_FILE": False,
    "FORCE_OVERWRITE": False,

    # --- Object Transforms ---
    "OBJ_LOCATION": (0.616392, -0.390241, -0.591646),
    "OBJ_ROTATION": (466.067, -2.68728, -306.609),
    "OBJ_SCALE": (0.006976, 0.006976, 0.006976),

    # --- Render ---
    "IMG_RES_X": 2000, "IMG_RES_Y": 2000,
    "NUM_SAMPLES": 100, "EXPOSURE": 1.5,
    "SUBDIVISION_LEVEL": 2,
    "DECIMATE_RATIO": 1.0,  # 1.0 = no decimation, 0.5 = keep 50% faces

    # --- Camera ---
    "CAM_LOCATION": (-1.9494, 1.5553, 0.71451),
    "LOOK_AT": (0, 0, 0),
    "FOCAL_LENGTH": 45,

    # --- Lighting ---
    "LIGHT_ANGLE": (-17.5966, -47, -384),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05,
}
print("✅ Registered: armadillo_PLY")

In [ ]:
# @title 4.2 Config — RueMadame_PLY  (template)
PLY_CONFIGS["RueMadame_PLY"] = {
    "SAVE_BLEND_FILE": True,
    "FORCE_OVERWRITE": True,

    "OBJ_LOCATION": (-20.0591, -23.3345, -9.59289),
    "OBJ_ROTATION": (-0.746005, -0.51064, 144.282),
    "OBJ_SCALE": (0.200259, 0.200259, 0.200259),

    "IMG_RES_X": 2000, "IMG_RES_Y": 2000,
    "NUM_SAMPLES": 100, "EXPOSURE": 1.5,
    "SUBDIVISION_LEVEL": 2,
    "DECIMATE_RATIO": 1,  # keep 30% faces (for heavy meshes)

    "CAM_LOCATION": (-1.9494, 1.5553, 0.71451),
    "LOOK_AT": (0, 0, 0),
    "FOCAL_LENGTH": 45,

    "LIGHT_ANGLE": (40.4034, -48, -396),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05,
}
print("✅ Registered: RueMadame_PLY")

In [ ]:
#@title 5. Master Config — Select & Write config.py

RENDER_TYPES = [
    # "armadillo_PLY",
    "RueMadame_PLY",
]

# Validate
for rt in RENDER_TYPES:
    assert rt in PLY_CONFIGS, f"❌ '{rt}' not in PLY_CONFIGS. Run its config cell first!"
print(f"Will render {len(RENDER_TYPES)} type(s): {RENDER_TYPES}")

# --- GCS + Kaggle paths ---
SHARED = {
    "GCS_BUCKET_NAME": "pyblender-render-farm",
    "GCS_INPUT_PREFIX": "PointCloud/plyFormat",
    "GCS_OUTPUT_PREFIX": "RenderImages/plyFormat",
    "GCS_KEY_PATH": "/tmp/gcs_service_account.json",
    "KAGGLE_OUTPUT_DIR": "/kaggle/working/RenderImages",
    "LOCAL_DATA_DIR": "/kaggle/working/local_data",
}

def _ts(v):
    if isinstance(v, tuple): return '(' + ', '.join(str(x) for x in v) + ')'
    if isinstance(v, list): return '[' + ', '.join(str(x) for x in v) + ']'
    return repr(v)

with open('config.py', 'w') as f:
    f.write('# AUTO-GENERATED by Master Config cell\n\n')
    f.write(f'RENDER_TYPES = {repr(RENDER_TYPES)}\n\n')
    # Shared
    f.write('SHARED = {\n')
    for ck, cv in SHARED.items():
        f.write(f'    "{ck}": {_ts(cv)},\n')
    f.write('}\n\n')
    # Per-dataset configs
    f.write('ALL_CONFIGS = {\n')
    for k in PLY_CONFIGS:
        f.write(f'    "{k}": {{\n')
        for ck, cv in PLY_CONFIGS[k].items():
            f.write(f'        "{ck}": {_ts(cv)},\n')
        f.write('    },\n')
    f.write('}\n')
print("✅ config.py written!")

In [ ]:
#@title 6. Download PLY Data from GCS
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/tmp/gcs_service_account.json"
client = storage.Client()
bucket = client.bucket("pyblender-render-farm")

for pc_type in RENDER_TYPES:
    gcs_prefix = f"{SHARED['GCS_INPUT_PREFIX']}/{pc_type}/"
    local_dir = os.path.join(SHARED['LOCAL_DATA_DIR'], pc_type)
    os.makedirs(local_dir, exist_ok=True)

    blobs = list(bucket.list_blobs(prefix=gcs_prefix))
    print(f"\n--- {pc_type}: {len(blobs)} files on GCS ---")

    for blob in blobs:
        fname = os.path.basename(blob.name)
        if not fname: continue
        dest = os.path.join(local_dir, fname)
        if not os.path.exists(dest):
            blob.download_to_filename(dest)
            print(f"  Downloaded: {fname}")
        else:
            print(f"  Exists: {fname}")

print("\n✅ All data ready!")

In [ ]:
%%writefile render_ply.py

import os, sys, time, threading, queue
import numpy as np

if '/kaggle/working' not in sys.path:
    sys.path.append('/kaggle/working')
from config import ALL_CONFIGS, SHARED, RENDER_TYPES

import bpy
import blendertoolbox as bt
import wandb


def readPLY_large(filePath, location, rotation_euler, scale):
    """Read PLY using plyfile + foreach_set — bypasses Blender's C++ importer
    that crashes with std::length_error on large meshes."""
    from plyfile import PlyData
    import math

    print(f"PLY import of '{os.path.basename(filePath)}' via plyfile...")
    t0 = time.time()
    plydata = PlyData.read(filePath)
    vertex = plydata['vertex']
    verts = np.column_stack([vertex['x'], vertex['y'], vertex['z']]).astype(np.float32)
    num_verts = len(verts)

    faces = None
    num_faces = 0
    if 'face' in plydata:
        face_el = plydata['face']
        if len(face_el) > 0:
            raw = face_el['vertex_indices']
            faces = [np.asarray(f) for f in raw]
            num_faces = len(faces)

    # Create Blender mesh using foreach_set (fast C-level transfer)
    mesh_data = bpy.data.meshes.new(os.path.basename(filePath))
    mesh_data.vertices.add(num_verts)
    mesh_data.vertices.foreach_set("co", verts.flatten())

    if faces and num_faces > 0:
        loop_totals = np.array([f.shape[0] for f in faces], dtype=np.int32)
        loop_starts = np.zeros(num_faces, dtype=np.int32)
        loop_starts[1:] = np.cumsum(loop_totals[:-1])
        all_loops = np.concatenate(faces).astype(np.int32)

        mesh_data.loops.add(len(all_loops))
        mesh_data.polygons.add(num_faces)
        mesh_data.loops.foreach_set("vertex_index", all_loops)
        mesh_data.polygons.foreach_set("loop_start", loop_starts)
        mesh_data.polygons.foreach_set("loop_total", loop_totals)

    mesh_data.update()
    mesh_data.validate()

    obj = bpy.data.objects.new(os.path.basename(filePath), mesh_data)
    bpy.context.collection.objects.link(obj)
    bpy.context.view_layer.objects.active = obj
    obj.select_set(True)

    x = rotation_euler[0] / 180.0 * math.pi
    y = rotation_euler[1] / 180.0 * math.pi
    z = rotation_euler[2] / 180.0 * math.pi
    obj.location = location
    obj.rotation_euler = (x, y, z)
    obj.scale = scale
    bpy.context.view_layer.update()

    elapsed = time.time() - t0
    print(f"PLY import of '{os.path.basename(filePath)}' took {elapsed*1000:.1f} ms "
          f"({num_verts} verts, {num_faces} faces)")
    return obj


class AsyncGCSUploader:
    def __init__(self, bucket):
        self.bucket = bucket
        self.queue = queue.Queue()
        self.uploads = 0
        self.failures = 0
        self._stop = False
        if bucket is not None:
            self.thread = threading.Thread(target=self._worker, daemon=True)
            self.thread.start()
        else:
            self.thread = None

    def _worker(self):
        while not self._stop or not self.queue.empty():
            try:
                local_path, gcs_path = self.queue.get(timeout=1)
                try:
                    blob = self.bucket.blob(gcs_path)
                    blob.upload_from_filename(local_path)
                    self.uploads += 1
                    print(f"  >> Uploaded to gs://{SHARED['GCS_BUCKET_NAME']}/{gcs_path}")
                except Exception as e:
                    self.failures += 1
                    print(f"  >> GCS upload failed: {e}")
                self.queue.task_done()
            except queue.Empty:
                continue

    def upload(self, local_path, gcs_path):
        if self.bucket is not None:
            self.queue.put((local_path, gcs_path))

    def wait_and_stop(self):
        if self.thread is not None:
            self.queue.join()
            self._stop = True
            self.thread.join(timeout=10)


def setup_gcs():
    try:
        from google.cloud import storage
        key_path = SHARED.get("GCS_KEY_PATH", "/tmp/gcs_service_account.json")
        if not os.path.exists(key_path):
            print(f"  GCS key not found at {key_path}")
            return None
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
        client = storage.Client()
        bucket = client.bucket(SHARED["GCS_BUCKET_NAME"])
        try:
            next(bucket.list_blobs(max_results=1), None)
            print(f"  GCS connected: gs://{SHARED['GCS_BUCKET_NAME']}")
        except Exception:
            print("  WARNING: Could not verify bucket.")
        return bucket
    except Exception as e:
        print(f"  ERROR setting up GCS: {e}")
        return None


def setup_cpu_render():
    """Configure Blender for CPU rendering with all available cores."""
    scene = bpy.context.scene
    scene.render.engine = 'CYCLES'
    scene.cycles.device = 'CPU'
    scene.render.threads_mode = 'AUTO'
    scene.cycles.use_denoising = True
    try:
        scene.cycles.denoiser = 'OPENIMAGEDENOISE'
    except Exception:
        pass
    scene.render.use_persistent_data = True
    try: scene.cycles.tile_size = 64
    except Exception: pass

    import multiprocessing
    print(f"  CPU render mode: {multiprocessing.cpu_count()} cores")
    print(f"  Denoiser: OpenImageDenoise (CPU)")


def render_single(CFG, meshPath, outputPath):
    bt.blenderInit(CFG["IMG_RES_X"], CFG["IMG_RES_Y"], CFG["NUM_SAMPLES"], CFG["EXPOSURE"])
    setup_cpu_render()

    # Use custom importer instead of bt.readMesh to avoid C++ crash
    mesh = readPLY_large(meshPath, CFG["OBJ_LOCATION"], CFG["OBJ_ROTATION"], CFG["OBJ_SCALE"])
    bpy.ops.object.shade_smooth()

    if CFG["SUBDIVISION_LEVEL"] > 0:
        bt.subdivision(mesh, level=CFG["SUBDIVISION_LEVEL"])
    meshC = bt.colorObj(bt.derekBlue, 0.5, 1.0, 1.0, 0.0, 0.0)
    subC = bt.colorObj(bt.derekBlue, 0.5, 2.0, 1.0, 0.0, 1.0)
    bt.setMat_ceramic(mesh, meshC, subC)
    mat = bpy.context.object.active_material
    nodes = mat.node_tree.nodes
    links = mat.node_tree.links
    mix_shader = next(
        (n for n in nodes if n.type == 'MIX_SHADER'
         and any(o.is_linked and o.links[0].to_node.type == 'OUTPUT_MATERIAL'
                 for o in n.outputs)), None)
    if mix_shader:
        if mix_shader.inputs['Fac'].is_linked:
            for link in mix_shader.inputs['Fac'].links: links.remove(link)
        if mix_shader.inputs[2].is_linked:
            for link in mix_shader.inputs[2].links: links.remove(link)
    cam = bt.setCamera(CFG["CAM_LOCATION"], CFG["LOOK_AT"], CFG["FOCAL_LENGTH"])
    sun = bt.setLight_sun(CFG["LIGHT_ANGLE"], CFG["LIGHT_STRENGTH"], CFG["SHADOW_SOFTNESS"])
    bt.setLight_ambient(color=CFG["AMBIENT_COLOR"])
    bt.shadowThreshold(alphaThreshold=CFG["SHADOW_THRESHOLD"], interpolationMode='CARDINAL')

    bpy.context.scene.use_nodes = True
    tree = bpy.context.scene.node_tree
    tree.nodes.clear()
    rl = tree.nodes.new('CompositorNodeRLayers')
    co = tree.nodes.new('CompositorNodeComposite')
    rl.location = (-300, 0)
    co.location = (300, 0)
    tree.links.new(rl.outputs['Image'], co.inputs['Image'])

    if CFG["SAVE_BLEND_FILE"]:
        bpy.ops.wm.save_mainfile(filepath=outputPath.replace(".png", ".blend"))
    bt.renderImage(outputPath, cam)


def render_all():
    print("--- Setting up Google Cloud Storage ---")
    gcs_bucket = setup_gcs()
    uploader = AsyncGCSUploader(gcs_bucket)

    for pc_type in RENDER_TYPES:
        CFG = ALL_CONFIGS[pc_type]
        print(f"\n{'='*60}")
        print(f"  RENDERING PLY: {pc_type} (TPU Runtime — CPU Mode)")
        print(f"{'='*60}\n")

        local_dir = os.path.join(SHARED["LOCAL_DATA_DIR"], pc_type)
        output_dir = os.path.join(SHARED["KAGGLE_OUTPUT_DIR"], "plyFormat", pc_type)
        os.makedirs(output_dir, exist_ok=True)

        if not os.path.exists(local_dir):
            print(f"ERROR: Input directory not found: {local_dir}")
            continue

        ply_files = sorted([f for f in os.listdir(local_dir) if f.endswith('.ply')])
        total = len(ply_files)
        print(f"Found {total} .ply files to render.\n")

        wandb.init(project="pyblender-render-farm", name=f"PLY_{pc_type}_TPU_CPU", config=CFG, reinit=True)

        completed = 0
        for idx, filename in enumerate(ply_files, 1):
            meshPath = os.path.join(local_dir, filename)
            out_name = filename.replace(".ply", ".png")
            outputPath = os.path.join(output_dir, out_name)
            gcs_blob_path = f"{SHARED['GCS_OUTPUT_PREFIX']}/{pc_type}/{out_name}"

            if gcs_bucket and not CFG["FORCE_OVERWRITE"]:
                blob = gcs_bucket.blob(gcs_blob_path)
                if blob.exists():
                    print(f"[{idx}/{total}] Already on GCS: {out_name}. Skipping...")
                    completed += 1
                    continue

            if os.path.exists(outputPath) and not CFG["FORCE_OVERWRITE"]:
                print(f"[{idx}/{total}] Exists locally: {out_name}. Skipping...")
                completed += 1
                continue

            start_time = time.time()
            print(f"[{idx}/{total}] Rendering [{filename}]...")
            render_single(CFG, meshPath, outputPath)
            uploader.upload(outputPath, gcs_blob_path)

            duration = time.time() - start_time
            completed += 1
            pct = (completed / total) * 100
            wandb.log({
                "progress_percent": pct,
                "render_time_seconds": duration,
                "filename": filename,
                "gcs_uploads_total": uploader.uploads,
                "latest_render": wandb.Image(outputPath),
            })
            print(f"[{idx}/{total}] Done: {filename} in {duration:.1f}s [{pct:.0f}%]\n")

        wandb.finish()

    print("\n--- Waiting for remaining GCS uploads ---")
    uploader.wait_and_stop()
    print(f"\n{'='*50}")
    print(f"  ALL RENDERS COMPLETE!")
    print(f"  GCS uploads: {uploader.uploads} (failures: {uploader.failures})")
    print(f"{'='*50}\n")


if __name__ == "__main__":
    render_all()


In [ ]:
#@title 8. Start Rendering (CPU Mode)

!./blender-4.0.2-linux-x64/blender -b -P render_ply.py

In [ ]:
#@title 9. Upload remaining local renders to GCS
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/tmp/gcs_service_account.json"
client = storage.Client()
bucket = client.bucket("pyblender-render-farm")

local_base = "/kaggle/working/RenderImages/plyFormat"
gcs_base = "RenderImages/plyFormat"
uploaded = 0

for root, dirs, files in os.walk(local_base):
    for f in files:
        local_path = os.path.join(root, f)
        relative = os.path.relpath(local_path, local_base)
        gcs_path = f"{gcs_base}/{relative}"
        blob = bucket.blob(gcs_path)
        if not blob.exists():
            blob.upload_from_filename(local_path)
            uploaded += 1
            print(f"  Uploaded: {gcs_path}")

print(f"\nDone! Uploaded {uploaded} files.")